In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from psilia import get_console
from psilia.data.mcap import (
    McapTaker as Taker, 
    get_mcap_overview
)
from psilia.data.utils import (
    psi_glob, 
    load_yaml, 
    save_yaml,
)
from psilia.vision.camera import (
    CameraIntrinsics, 
    unproject, 
    project,
)
from psilia.transforms import (
    Transform, 
    TransformTree,
    CAM_ALONG_X
)
from psilia.plotting import RerunLogger
import jax
import jax.numpy as jnp
import cv2
import numpy as np
import matplotlib.pyplot as plt
console = get_console()

In [ ]:
# from psilia.plotting import RerunLogger

# rrl = RerunLogger("Depth-Baselines")

In [ ]:
console = get_console()
print = console.print

path = str(Path('~/workspace/data/rosbags/**/*.mcap').expanduser())
mcaps = psi_glob(path)
console.print(mcaps[["name", "path"]])

mcap = mcaps["path"][1]
overview = get_mcap_overview(mcap)
console.print(overview)

In [ ]:
taker = Taker(mcap, 
    topic_map={
        "/zed/zed_node/pose": "/pose", 
        "/zed/zed_node/left/camera_info": "/left_intr",
        "/zed/zed_node/left/image_rect_color": "/left_im",
        "/zed/zed_node/right/camera_info": "/right_intr",
        "/zed/zed_node/right/image_rect_color": "/right_im",
        "/zed/zed_node/depth/depth_registered": "/depth",
        "/zed/zed_node/confidence/confidence_map": "/conf"
    }, 
    schema_transforms = {
        "sensor_msgs/msg/CameraInfo": {
            "__node__": lambda d: CameraIntrinsics.from_camera_info_dict(d),
        },
        "sensor_msgs/msg/Image": {
            "__node__": lambda d: d["data"]
        },
        "geometry_msgs/msg/PoseStamped" : {
            "__node__": lambda d: Transform.from_dict(d),
        }
    },
    topic_transforms={
        "/tf_static": {"__node__": lambda d: TransformTree(d["transforms"], strict=False)},
        "/conf": {"__node__": lambda d: 1.-d["data"][:,:,None]/100},
    })

In [ ]:
intr, tf_static = taker.k[
    "/left_intr", "/tf_static"].i[0,0].t(0, sort_key="publish_time")

console.print(intr)
console.print(tf_static)

In [ ]:
tf01 = tf_static["zed_left_camera_optical_frame", "zed_right_camera_optical_frame"]
tf10 = tf01.inv()
B = tf01.t[0] # Baseline for depth from disparity
console.print(tf01)

In [ ]:

t = 5.0
im0, im1, depth, conf = taker.k["/left_im", "/right_im", "/depth", "/conf"].i[0,0,0,0].t(t)
im0 = im0[...,:3]
im1 = im1[...,:3]
console.inspect(im0=im0, im1=im1)

# ====================================
fig, axs = plt.subplots(1,4, figsize=(10,5))
axs[0].set_title("Left Image")
axs[0].imshow(im0)
axs[1].set_title("Right Image")
axs[1].imshow(im1)
axs[2].set_title("Depth")
axs[2].imshow(depth, vmin=0.2, vmax=3)
axs[3].set_title("Confidence")
axs[3].imshow(conf, vmin=0, vmax=1)

In [ ]:
from psilia.vision.camera import camera_from_screen_and_depth

In [ ]:
def compute_pixel_nbh(uv, radius=3):
    uvs = jnp.stack(jnp.meshgrid(
        jnp.arange(-radius, +radius+1), 
        jnp.arange(-radius, +radius+1), 
        indexing="ij"), -1).reshape(-1,2) + uv
    return uvs

In [ ]:
jax.lax.dynamic_slice?

In [ ]:
def to_gray(rgb):
    return jnp.dot(rgb[...,:3], jnp.array([0.2989, 0.5870, 0.1140]))

In [ ]:
uv = jnp.array([300,90]) + 0.5

t = 5.0
im0, im1, depth, conf = taker.k["/left_im", "/right_im", "/depth", "/conf"].i[0,0,0,0].t(t)
im0 = im0[...,:3]/255
im1 = im1[...,:3]/255
console.inspect(im0=im0, im1=im1)

# ====================================
fig, axs = plt.subplots(1,4, figsize=(10,5))
axs[0].set_title("Left Image")
axs[0].imshow(im0)
axs[0].scatter(uv[0], uv[1], color="red")
axs[1].set_title("Right Image")
axs[1].imshow(im1)
axs[2].set_title("Depth")
axs[2].imshow(depth, vmin=0.2, vmax=3)
axs[3].set_title("Confidence")
axs[3].imshow(conf, vmin=0, vmax=1)

In [ ]:
uv = jnp.array([300,90]) + 0.5

# def _f(z, uv):
z = jnp.array(1.1)

# uvs = compute_pixel_nbh(uv, radius=1)
# xs  = camera_from_screen_and_depth(uvs, z, intr)
# uvs_ = project(xs, intr)


x  = camera_from_screen_and_depth(uv, z, intr)
uv_,_ = project(x, intr)

r = 10

w0 = jax.lax.dynamic_slice(im0, (int(uv[1])-r, int(uv[0])-r, 0), (2*r+1, 2*r+1, 3))
w1 = jax.lax.dynamic_slice(im1, (int(uv_[1])-r, int(uv_[0])-r, 0), (2*r+1, 2*r+1, 3))

console.print(jnp.sum(jnp.linalg.norm(w0-w1, axis=-1)))

console.inspect(w0=w0, w1=w1)

fig, axs = plt.subplots(1,2, figsize=(10,5))
axs[0].set_title("Left Patch")
axs[0].imshow(w0)
axs[1].set_title("Right Patch")
axs[1].imshow(w1)


In [ ]:
coords = jnp.arange(2*r+1) - r  # [-r, ..., 0, ..., r]
gy, gx = jnp.meshgrid(coords, coords, indexing='ij')
gauss = jnp.exp(-(gx**2 + gy**2) / (2 * (r / 2.0)**2))
gauss = gauss / gauss.sum()

In [ ]:
key = jax.random.PRNGKey(0)

In [ ]:
# key,_=jax.random.split(key)
uv = jax.random.uniform(key, (2,), minval=jnp.array([0,0]), maxval=jnp.array([intr.w, intr.h]))


r = 5

coords = jnp.arange(2*r+1) - r  # [-r, ..., 0, ..., r]
gy, gx = jnp.meshgrid(coords, coords, indexing='ij')
gauss = jnp.exp(-(gx**2 + gy**2) / (2 * (r / 4.0)**2))
gauss = gauss / gauss.sum()




def _get_pair(uv, z):
    x  = camera_from_screen_and_depth(uv, z, intr)
    uv_,_ = project(tf10(x), intr)

    w0 = jax.lax.dynamic_slice(im0, (uv[1].astype(int)-r, uv[0].astype(int)-r, 0), (2*r+1, 2*r+1, 3))
    w1 = jax.lax.dynamic_slice(im1, (uv_[1].astype(int)-r, uv_[0].astype(int)-r, 0), (2*r+1, 2*r+1, 3))
    return w0, w1


def _f(z, uv):

    w0, w1 = _get_pair(uv, z)
    diff = jnp.sum(gauss*jnp.linalg.norm(w0 - w1, axis=-1))

    return diff, None

f = jax.vmap(_f, (0, None))


# zs = jnp.linspace(0.5, 2.0, 200)
zs = intr.fx*B/jnp.arange(200)
vs, _ = f(zs, uv)
console.inspect(zs=jnp.diff(zs), vs=vs)
console.print(depth[int(uv[1]), int(uv[0])])


i = jnp.argmin(vs)
w0, w1 = _get_pair(uv, zs[i])


def infer_depth(uv):
    zs = jnp.linspace(0.3, 2.0, 100)
    vs, _ = f(zs, uv)
    i = jnp.argmin(vs)
    return zs[i], jnp.std(vs)


# ====================
plt.figure(figsize=(5,2))
plt.plot(zs,vs)
plt.scatter(zs[i], vs[i], color='red')
plt.vlines(depth[int(uv[1]), int(uv[0])], vs.min(), vs.max(), color="red")

fig, axs = plt.subplots(1,2, figsize=(10,5))
axs[0].imshow(w0)
axs[1].imshow(w1)   

# ====================================
fig, axs = plt.subplots(1,4, figsize=(10,5))
axs[0].set_title("Left Image")
axs[0].imshow(im0)
axs[0].scatter(uv[0], uv[1], color="red")
axs[1].set_title("Right Image")
axs[1].imshow(im1)
axs[2].set_title("Depth")
axs[2].imshow(depth, vmin=0.2, vmax=3)
axs[3].set_title("Confidence")
axs[3].imshow(conf, vmin=0, vmax=1)

In [ ]:
plt.plot(zs, marker="o")

In [ ]:
from ipywidgets import interact, IntSlider


@interact(i=IntSlider(min=0, max=len(zs)-1, value=jnp.argmin(vs)))
def show(i):

    w0, w1 = _get_pair(uv, zs[i])
    # ====================
    plt.figure(figsize=(5,2))
    plt.plot(zs, vs)
    plt.scatter(zs[i], vs[i], color='red')
    plt.vlines(depth[int(uv[1]), int(uv[0])], vs.min(), vs.max(), color="red", linestyles="dashed")

    fig, axs = plt.subplots(1,3, figsize=(10,5))
    axs[0].imshow(w0)
    axs[1].imshow(w1)   
    axs[2].imshow(jnp.linalg.norm(w0 - w1, axis=-1), vmin=0, vmax=1.)   

In [ ]:
B, im0.shape, im0.dtype

In [ ]:
from psilia.vision.depth_cv import stereo_depth


dd = stereo_depth(
    (im0*255).astype(np.uint8), 
    (im1*255).astype(np.uint8), 
    intr, 
    B,
    num_disparities = 128,
    block_size = 11,
    min_disparity = 0,
)
# dd = stereo_depth((im0*255).astype(jnp.uint8), (im1*255).astype(jnp.uint8), intr, B)

console.inspect(dd=dd)
plt.imshow(dd, vmin=0.1, vmax=3)

In [ ]:
key,_=jax.random.split(key)
uvs = jax.random.uniform(key, (100_000, 2), minval=jnp.array([0,0]), maxval=jnp.array([intr.w, intr.h]))

# mask = uvs[:,0] > 120
# uvs=uvs[mask]

inferred, stds = jax.lax.map(infer_depth,uvs)
# inferred = dd[uvs[:,1].astype(int), uvs[:,0].astype(int)]
actual = depth[uvs[:,1].astype(int), uvs[:,0].astype(int)]
# actual = dd[uvs[:,1].astype(int), uvs[:,0].astype(int)]

errs = inferred - actual
console.print(jnp.sum(jnp.abs(errs)<= 0.2)/errs.shape[0])

# ==============
plt.figure(figsize=(20,5))
plt.gca().set_aspect('equal')
plt.xlim(-3,3)
plt.scatter(inferred - actual, stds, s=1, alpha=0.1)

In [ ]:
order = jnp.argsort(jnp.abs(errs))[:]

plt.imshow(im0)
plt.scatter(uvs[order,0], uvs[order,1], c=jnp.abs(errs)[order], s=1, vmin=0, vmax=0.5, alpha=0.5)

In [ ]:
errs = inferred - actual
mask = jnp.isfinite(errs)
plt.hist2d(errs[mask], stds[mask], bins=(np.linspace(-.5,.5,100), np.linspace(0,0.5,100)));

In [ ]:
from psilia.vision.camera import render_naively
xs_inf = camera_from_screen_and_depth(uvs, inferred, intr)

im = render_naively(xs_inf, inferred, intr, intr.image_shape, down=4, fill_value=0.0)

plt.imshow(im)

In [ ]:
from psilia.plotting import RerunLogger

rrl = RerunLogger("Depth-Baselines")

mask = (jnp.abs(errs)<= 0.2)

rrl.set_time(0)
rrl.log_points("xs", CAM_ALONG_X(xs_inf[mask]))

In [ ]:
dd = stereo_depth(
    (im0*255).astype(np.uint8), 
    (im1*255).astype(np.uint8), 
    intr, 
    B,
    num_disparities = 128,
    block_size = 11,
    min_disparity = 0,
)
# dd = stereo_depth((im0*255).astype(jnp.uint8), (im1*255).astype(jnp.uint8), intr, B)

xs = camera_from_screen_and_depth(uvs, dd[uvs[:,1].astype(int), uvs[:,0].astype(int)], intr)



rrl.set_time(1)
rrl.log_points("xs", CAM_ALONG_X(xs))